First make sure `Load_Meshcat = True` in base_interface.py

In [2]:
import sys
import os
# Directory Management
try:
    # Run in Terminal
    ROOT_DIR = os.path.dirname(os.path.abspath(__file__))
except:
    # Run in ipykernel & interactive
    ROOT_DIR = os.getcwd()
if ".." not in sys.path:
    sys.path.append("..")

from fast_legged_planner_py.robot_interface.elspidermini_robotinterface import ElSpiderMini_RobotInterface, FOOT_LINK_NAME
from fast_legged_planner_py.robot_interface.pin_IK import pinIK
import pinocchio as pin
import numpy as np


urdf_file = os.path.join(os.path.dirname(ROOT_DIR), "model", "ElSpider_Mini", "urdf", "el_mini.urdf")
pack_dirs = [os.path.dirname(os.path.dirname(ROOT_DIR))]
el_mini = ElSpiderMini_RobotInterface(urdf_file, pack_dirs)

In [2]:
# el_mini.print_frames()
q = el_mini.robot.q0
el_mini.viz.display(q)
# el_mini.get_frameid("LF_FOOT")
# el_mini.robot.framePlacement(q, el_mini.get_frameid("LF_FOOT"))

In [4]:
# Display
q0 = el_mini.robot.q0
q_leg = [0, 0, 0]
q = el_mini.get_full_q(q_leg, 1)
el_mini.viz.display(q)
# el_mini.vis_collision_model(q0)
for link in FOOT_LINK_NAME:
    print(el_mini.get_frame_placement(q,link).translation)
# el_mini.robot.computeFrameJacobian
# print(el_mini.robot.data.oMi[3].translation)

[ 0.35350208 -0.22998902 -0.1360558 ]
[ 0.05350208 -0.28998902 -0.1360558 ]
[-0.35349792 -0.22998928 -0.1360562 ]
[ 0.35350208  0.2299882  -0.1360569 ]
[ 0.05350208  0.2899882  -0.1360569 ]
[-0.35350133  0.22998794 -0.13605704]


In [3]:
## Inverse Kinematics (IKFast single leg)
leg_num = 0
q_leg = el_mini.IKFast_foot(leg_num, np.array([0.3, 0.3,-0.2]))
# print(q_leg)
el_mini.viz.display(el_mini.get_full_q(q_leg,leg_num))

In [4]:
## Inverse Kinematics (IKFast all legs)
import time
use_ikfast = True
check_valid = False
fall_back = False
ray_approx = False
viz = False
sample_num = 1000
amp = 0.1
h = -0.13
index = None
sphere_size = 0.01
el_mini.viz_clear()
if use_ikfast:
    IKs = el_mini.IKFast_foots
    IK = el_mini.IKFast_foot
else:
    IKs = lambda target_list, valid_check, fall_back, ray_approx: el_mini.IK_foots(target_list)
    IK = lambda index, target, valid_check, fall_back, ray_approx: el_mini.IK_foot(index, target)
def random_target():
    return (np.random.rand(3)-0.5)*amp

errcnt = 0
start = time.perf_counter()
for theta in np.linspace(0, 2*np.pi, sample_num):
    target_list = [np.array([-amp*np.sin(theta), amp*np.cos(theta), h]) for i in range(6)]
    # target_list = [random_target() for i in range(6)]
    try:
        if index is not None:
            q = IK(index, np.array(target_list[index]),
                   valid_check=check_valid, fall_back=fall_back, ray_approx=ray_approx)
            q = el_mini.get_full_q(q, index)
            if viz:
                el_mini.viz_add_sphere(target_list[index], sphere_size)
        else:
            q = IKs(target_list, valid_check=check_valid, fall_back=fall_back, ray_approx=ray_approx)
            if viz:
                for pos in target_list:
                    el_mini.viz_add_sphere(pos, sphere_size)
        el_mini.viz.display(q)
    except Exception as e:
        # print(e)
        errcnt += 1
        pass
print("cost %s second" % (time.perf_counter() - start))
print("error count: %s" % errcnt)

cost 5.938517460017465 second
error count: 192


In [19]:
# from fast_legged_planner_py.robot_interface.pyikfast import pyikfast_hitspider as ik
from fast_legged_planner_py.robot_interface.pyikfast import pyikfast_el_mini as ik
ik.IKFast_trans3D([0.35349792, -0.22998928,-0])

[]